# Hybrid CNN+GRU — k-fold cross-validation

v1 hybrid (notebook 07) overfit dramatically on the OOD test set: best val MAE 4.14% but test MAE 11.63%. With only 45 val sequences and 6 OOD test batteries, the single-split signal is too noisy to trust.

This notebook addresses both issues:
1. **5-fold CV over the 22 train-pool batteries** (matching `00_kfold_baselines.ipynb`'s SOH-stratified split). Reports mean ± std val MAE — comparable to the existing baseline table.
2. **Smaller model**: `cnn_filters=16, gru_hidden=32, k_steps=8, dropout=0.5` → ~45K params (vs 168K in v1). Capacity now matches the ~260 sequences/fold training set.

Hyperparameters copied from cycle-history notebooks (LR=5e-4, batch=16, max 100 epochs, patience=15, grad clip=1.0).

In [ ]:
import json
import math
import os
import sys
import time

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import DEFAULT_TEST_BATTERIES
from src.dataset_hybrid import HybridDataset
from src.models.hybrid import SOHHybrid
from src.utils import set_all_seeds

H5_PATH      = '../data/processed/sequences.h5'
METRICS_PATH = '../results/metrics/hybrid_kfold_summary.json'
CKPT_DIR     = '../checkpoints'

# k-fold and training config
K              = 5
WINDOW         = 10
K_STEPS        = 8
CNN_FILTERS    = 16
GRU_HIDDEN     = 32
DROPOUT        = 0.5

LR             = 5e-4
WEIGHT_DECAY   = 1e-4
BATCH_SIZE     = 16
MAX_EPOCHS     = 100
PATIENCE       = 15
CLIP_GRAD      = 1.0
SEED           = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
set_all_seeds(SEED)
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')

In [ ]:
# Build SOH-stratified k-fold battery splits — same logic as
# BatterySOHDataset.create_kfold_splits, but we only need the battery lists
# (HybridDataset partitions internally by battery name).
with h5py.File(H5_PATH, 'r') as f:
    bids = np.array([b.decode('ascii') for b in f['battery_id'][:]])
    y_all = f['y'][:]

test_set = set(DEFAULT_TEST_BATTERIES)
pool = sorted([b for b in set(bids.tolist()) if b not in test_set])
mean_soh = {b: float(y_all[bids == b].mean()) for b in pool}

ordered = sorted(pool, key=lambda b: mean_soh[b])
rng = np.random.default_rng(SEED)
blocks = [ordered[i:i+K] for i in range(0, len(ordered), K)]
for blk in blocks:
    rng.shuffle(blk)

fold_val_batteries: list[list[str]] = [[] for _ in range(K)]
for blk in blocks:
    for i, b in enumerate(blk):
        fold_val_batteries[i].append(b)

for fi, vb in enumerate(fold_val_batteries):
    vmean = np.mean([mean_soh[b] for b in vb])
    print(f'Fold {fi}: val={vb}  (mean SOH {vmean:.1f}%)  '
          f'train={len([b for b in pool if b not in set(vb)])} batteries')

In [ ]:
# Preload X once (shared across all 5 folds)
t0 = time.time()
X_mem = HybridDataset.preload_X(H5_PATH)
print(f'Preloaded X: {X_mem.shape}  ({X_mem.nbytes / 1024**3:.2f} GB)  '
      f'in {time.time()-t0:.1f}s')

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(DEVICE)).squeeze(1).cpu())
            targets.append(yb)
    p = torch.cat(preds)   * 100.0
    t = torch.cat(targets) * 100.0
    mae  = (p - t).abs().mean().item()
    rmse = math.sqrt(((p - t) ** 2).mean().item())
    return mae, rmse


def train_one_fold(fold_idx: int, val_batteries: list[str]) -> dict:
    set_all_seeds(SEED + fold_idx, deterministic=False)

    # Build datasets — HybridDataset splits internally by battery name.
    train_ds = HybridDataset(H5_PATH, window=WINDOW, k_steps=K_STEPS, split='train',
                              val_batteries=val_batteries,
                              preloaded_X=X_mem)
    val_ds   = HybridDataset(H5_PATH, window=WINDOW, k_steps=K_STEPS, split='val',
                              val_batteries=val_batteries,
                              norm_mean=train_ds.norm_mean,
                              norm_std=train_ds.norm_std,
                              deterministic=True, preloaded_X=X_mem)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = SOHHybrid(window=WINDOW, k_steps=K_STEPS,
                       cnn_filters=CNN_FILTERS, gru_hidden=GRU_HIDDEN,
                       dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_val_mae   = float('inf')
    best_val_rmse  = float('inf')
    best_epoch     = 0
    best_state     = None
    no_improve     = 0
    t_start        = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb).squeeze(1), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            optimizer.step()
            running += loss.item()
        train_loss = running / max(len(train_loader), 1)

        val_mae, val_rmse = evaluate(model, val_loader)
        scheduler.step(val_mae)

        if val_mae < best_val_mae:
            best_val_mae  = val_mae
            best_val_rmse = val_rmse
            best_epoch    = epoch
            best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    elapsed = time.time() - t_start
    return {
        'fold':           fold_idx,
        'val_batteries':  list(val_batteries),
        'best_val_mae':   best_val_mae,
        'best_val_rmse':  best_val_rmse,
        'best_epoch':     best_epoch,
        'epochs_run':     epoch,
        'elapsed_sec':    elapsed,
        'n_train_seqs':   len(train_ds),
        'n_val_seqs':     len(val_ds),
        'best_state':     best_state,
    }

In [ ]:
# Run all 5 folds
all_results: list[dict] = []

print(f'{"Fold":>4}  {"#train":>7}  {"#val":>5}  {"BestMAE%":>9}  {"BestRMSE%":>10}  {"BestEp":>7}  {"Run":>5}  {"Time":>6}')
print('-' * 70)
for fi, vb in enumerate(fold_val_batteries):
    r = train_one_fold(fi, vb)
    all_results.append(r)
    print(f'{r["fold"]:>4}  {r["n_train_seqs"]:>7}  {r["n_val_seqs"]:>5}  '
          f'{r["best_val_mae"]:>9.4f}  {r["best_val_rmse"]:>10.4f}  '
          f'{r["best_epoch"]:>7}  {r["epochs_run"]:>5}  {r["elapsed_sec"]:>5.0f}s')
    # Save checkpoint per fold
    torch.save({'fold': fi, 'val_mae': r['best_val_mae'],
                'model_state_dict': r['best_state']},
               os.path.join(CKPT_DIR, f'SOHHybrid_kfold{fi}.pt'))

# Aggregate
maes  = np.array([r['best_val_mae']  for r in all_results])
rmses = np.array([r['best_val_rmse'] for r in all_results])
summary = {
    'n_folds':       int(K),
    'val_mae_mean':  float(maes.mean()),
    'val_mae_std':   float(maes.std(ddof=1)),
    'val_mae_min':   float(maes.min()),
    'val_mae_max':   float(maes.max()),
    'val_rmse_mean': float(rmses.mean()),
    'val_rmse_std':  float(rmses.std(ddof=1)),
    'config': dict(window=WINDOW, k_steps=K_STEPS, cnn_filters=CNN_FILTERS,
                   gru_hidden=GRU_HIDDEN, dropout=DROPOUT,
                   lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE),
    'runs': [{k: v for k, v in r.items() if k != 'best_state'} for r in all_results],
}
with open(METRICS_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print()
print(f'Hybrid CNN+GRU (k-fold, small): val MAE {maes.mean():.4f} ± {maes.std(ddof=1):.4f}%   '
      f'(min {maes.min():.2f}%, max {maes.max():.2f}%)')
print(f'Summary saved to {METRICS_PATH}')

In [ ]:
# Final test evaluation: pick the best-performing fold's checkpoint and
# evaluate on the OOD test set. (The clean alternative — train on full pool
# until a median best-epoch — would require an extra training run; this gives
# a representative test number using one of the 5 already-trained models.)
best_fold = int(np.argmin(maes))
ckpt_path = os.path.join(CKPT_DIR, f'SOHHybrid_kfold{best_fold}.pt')
ckpt = torch.load(ckpt_path, map_location=DEVICE)

best_val_bats = fold_val_batteries[best_fold]
train_ds = HybridDataset(H5_PATH, window=WINDOW, k_steps=K_STEPS, split='train',
                          val_batteries=best_val_bats, preloaded_X=X_mem)
test_ds  = HybridDataset(H5_PATH, window=WINDOW, k_steps=K_STEPS, split='test',
                          val_batteries=best_val_bats,
                          norm_mean=train_ds.norm_mean, norm_std=train_ds.norm_std,
                          deterministic=True, preloaded_X=X_mem)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = SOHHybrid(window=WINDOW, k_steps=K_STEPS,
                   cnn_filters=CNN_FILTERS, gru_hidden=GRU_HIDDEN,
                   dropout=DROPOUT).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
test_mae, test_rmse = evaluate(model, test_loader)
print(f'Best fold by val MAE: {best_fold}  (val MAE {ckpt["val_mae"]:.4f}%)')
print(f'Test MAE on OOD set:  {test_mae:.4f}%')
print(f'Test RMSE on OOD set: {test_rmse:.4f}%')

In [ ]:
# Comparison table
print('=' * 88)
print(f'{"Model":<36}  {"Input Shape":<22}  {"MAE%":>10}  {"RMSE%":>10}')
print('-' * 88)
print(f'{"MLP (k-fold)":<36}  {"20 stats/step":<22}  {"11.62":>10}  {"16.13":>10}')
print(f'{"CNN (k-fold)":<36}  {"(301,3)/step":<22}  {" 7.46":>10}  {"12.64":>10}')
print(f'{"LSTM single-step (k-fold)":<36}  {"(301,3)/step":<22}  {"10.88":>10}  {"15.61":>10}')
print(f'{"Transformer single-step (k-fold)":<36}  {"(301,3)/step":<22}  {"10.25":>10}  {"15.23":>10}')
print(f'{"LSTM cycle-history":<36}  {"(10,15)/blocks":<22}  {" 8.24":>10}  {"10.43":>10}')
print(f'{"Transformer cycle-history":<36}  {"(10,15)/blocks":<22}  {" 9.68":>10}  {"12.66":>10}')
print(f'{"Hybrid CNN+GRU (single split, v1)":<36}  {"(10,32,301,3)":<22}  {"11.63":>10}  {"15.84":>10}')
mae_str  = f'{maes.mean():.2f} ± {maes.std(ddof=1):.2f}'
rmse_str = f'{rmses.mean():.2f} ± {rmses.std(ddof=1):.2f}'
print(f'{"Hybrid CNN+GRU (k-fold, small)":<36}  {f"(10,{K_STEPS},301,3)":<22}  {mae_str:>10}  {rmse_str:>10}')
print('=' * 88)